## Step 0 — Setup & model loading

Imports, repo root, FastF1 cache. `RadioAgentCFG` loads the three N24 NLP models
(RoBERTa-base sentiment, SetFit intent, BERT-large NER) plus the RCM rule-based parser.
Module-level globals `PIPELINE` and `SESSION_META` are populated by `setup_session()`.

**Models loaded:**
- `data/models/nlp/` — roberta_sentiment.pt, setfit_intent.pt, bert_ner.pt
- `data/models/nlp/pipeline_config_v1.json` — label maps and thresholds

**Input schemas defined here:**
- `RadioMessage(driver, lap, text, timestamp=None)` — transcribed or mocked radio
- `RCMEvent` parsed from FastF1 `session.race_control_messages`


---

In [ ]:
import json, sys, re, time, warnings
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional

import torch
import numpy as np
import pandas as pd
import fastf1
import transformers.training_args as _tra

warnings.filterwarnings("ignore")

if not hasattr(_tra, "default_logdir"):
    _tra.default_logdir = lambda: "runs"

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

fastf1.Cache.enable_cache(str(repo_root / "data" / "cache" / "fastf1"))

# Module-level globals — populated by setup_session()
LAPS:         pd.DataFrame = pd.DataFrame()
RCM_DF:       pd.DataFrame = pd.DataFrame()
SESSION_META: dict         = {}


In [ ]:
@dataclass
class RadioAgentCFG:
    """Runtime configuration for the Radio Agent.

    Loads the three N24 NLP models (sentiment, intent, NER) inline and assembles
    the pipeline dict consumed by run_pipeline(). Device is auto-detected from
    CUDA availability so the notebook runs on CPU when no GPU is present.

    model_name:
        LM Studio local model identifier for the ReAct agent LLM. Must match
        the model currently loaded in LM Studio.
    device:
        Torch device for all three NLP models. Resolved at init time from
        CUDA availability — same device is used for sentiment, NER, and
        inference tensors inside the tool functions.
    pipeline:
        Dict assembled in __post_init__ with keys sentiment_model,
        sentiment_tokenizer, intent_model, ner_model, ner_tokenizer,
        ner_label2id, ner_id2label. Same schema as N24's build_pipeline().
        Stored here so tool functions can call run_pipeline(text, CFG.pipeline).
    alert_intents:
        Intent labels that trigger an alert entry in RadioOutput. PROBLEM and
        WARNING from N21 SetFit are the two signals relevant for N31 escalation.
    intent_names:
        Ordered label list matching N21 SetFit training output. Used by tool
        functions to decode predict_proba() indices into human-readable labels.
    sentiment_labels:
        Ordered label list matching N20 RoBERTa training output (neg/neu/pos).
    ner_max_len:
        Maximum token length for BERT-large NER tokenisation. Must match N22
        training config to avoid truncation artifacts on long radio messages.
    """

    model_name:       str   = "local-model"
    device:           str   = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    alert_intents:    tuple = ("PROBLEM", "WARNING")
    intent_names:     tuple = ("INFORMATION", "PROBLEM", "ORDER", "WARNING", "QUESTION")
    sentiment_labels: tuple = ("negative", "neutral", "positive")
    ner_max_len:      int   = 128
    pipeline:         dict  = field(init=False)

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        _nlp_dir = root / "data" / "models" / "nlp"

        # --- Sentiment model (N20): RoBERTa-base fine-tuned, 3-class ---
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        print("Loading sentiment model (N20)...")
        _s_dir  = _nlp_dir / "bert_sentiment_v1"
        s_tok   = AutoTokenizer.from_pretrained("roberta-base")
        s_model = AutoModelForSequenceClassification.from_pretrained(
            "roberta-base", num_labels=3
        )
        _s_sd = torch.load(
            _s_dir / "best_roberta_sentiment_model.pt",
            map_location=self.device, weights_only=False,
        )
        if any(k.startswith("model.") for k in _s_sd):
            _s_sd = {k[len("model."):]: v for k, v in _s_sd.items()}
        s_model.load_state_dict(_s_sd)
        s_model = s_model.to(self.device).eval()

        # --- Intent model (N21): SetFit + ModernBERT, 5-class ---
        from setfit import SetFitModel
        print("Loading intent model   (N21)...")
        _i_dir  = _nlp_dir / "intent_setfit_modernbert_v1"
        i_model = SetFitModel.from_pretrained(str(_i_dir))

        # --- NER model (N22): BERT-large CoNLL-03 BIO, 19-label ---
        from transformers import BertForTokenClassification
        print("Loading NER model      (N22)...")
        _n_dir  = _nlp_dir / "ner_v1" / "bert_bio_v1"
        _n_cfg  = json.loads((_n_dir / "model_config.json").read_text())
        n_l2i   = _n_cfg["label2id"]
        n_i2l   = {int(k): v for k, v in _n_cfg["id2label"].items()}
        _n_base = _n_cfg.get(
            "model_name", "dbmdz/bert-large-cased-finetuned-conll03-english"
        )
        n_tok   = AutoTokenizer.from_pretrained(str(_n_dir), use_fast=True)
        n_model = BertForTokenClassification.from_pretrained(
            _n_base, num_labels=len(n_l2i), ignore_mismatched_sizes=True
        )
        _n_sd = torch.load(
            _n_dir / "bert_bio_state_dict.pt",
            map_location=self.device, weights_only=False,
        )
        n_model.load_state_dict(_n_sd)
        n_model = n_model.to(self.device).eval()

        print("All models loaded.")
        self.pipeline = {
            "sentiment_model":     s_model,
            "sentiment_tokenizer": s_tok,
            "intent_model":        i_model,
            "ner_model":           n_model,
            "ner_tokenizer":       n_tok,
            "ner_label2id":        n_l2i,
            "ner_id2label":        n_i2l,
        }


In [ ]:
CFG = RadioAgentCFG()

print(f"\nModel           : {CFG.model_name}")
print(f"Device          : {CFG.device}")
print(f"Intent labels   : {CFG.intent_names}")
print(f"Sentiment labels: {CFG.sentiment_labels}")
print(f"NER max len     : {CFG.ner_max_len}")
print(f"Alert intents   : {CFG.alert_intents}")
print(f"Pipeline keys   : {list(CFG.pipeline.keys())}")
